# MLE_codes

本笔记是 `MLE_lec.qmd` 的配套素材工厂，负责完成两类任务：

- 生成本章讲义所需的图形，并统一保存到 `./figs/` 文件夹；
- 生成后续案例分析和后续章节可复用的模拟数据，并保存到 `./data/` 文件夹。

本笔记的设计原则有两点：

- 每段代码之前都先说明这段代码在讲义中的作用，而不是直接进入编程；
- 图形和数据文件采用统一命名规则，方便在 Quarto 讲义和后续 notebook 中直接调用。


## 0. 环境设置与输出路径

先导入本笔记需要的库，并创建 `figs/` 和 `data/` 两个输出文件夹。后续所有图形和数据都写入这两个目录。


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.optimize import minimize

import warnings
warnings.filterwarnings('ignore')
# 中文字体：Windows 用 SimHei，Linux 用 Noto Sans CJK SC 或 WenQuanYi Micro Hei
import platform
if platform.system() == 'Windows':
    FONT_FAMILY = 'SimHei'
else:
    # Linux/macOS 回退方案：优先尝试常见中文字体
    import matplotlib.font_manager as fm
    available = [f.name for f in fm.fontManager.ttflist]
    for candidate in ['Noto Sans CJK SC', 'WenQuanYi Micro Hei', 'Source Han Sans CN', 'SimHei']:
        if candidate in available:
            FONT_FAMILY = candidate
            break
    else:
        FONT_FAMILY = 'DejaVu Sans'  # 最终回退，中文将显示为方块


os.makedirs('./figs', exist_ok=True)
os.makedirs('./data', exist_ok=True)

plt.rcParams.update({
    'font.family': 'Noto Sans CJK JP',
    'axes.unicode_minus': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'figure.figsize': (6, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

COLOR_PRIMARY   = '#2C6BAC'
COLOR_SECONDARY = '#B8C500'
COLOR_NEUTRAL   = '#888888'
COLOR_FILL      = '#D6E8F7'
COLOR_ORANGE    = '#D97A2B'
COLOR_GREEN     = '#2C9A57'


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()

print('环境设置完成。')


环境设置完成。


## 1. 图 01：MLE 的完整分析链条

本图是整章的导航图，用来展示 MLE 的核心逻辑：从样本数据出发，经过分布假设、参数化、似然函数，再到参数估计与模型比较。该图会在 `MLE_lec.qmd` 的导言和小结中重复引用。


In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.axis('off')
labels = [
    '样本数据\n$\{y_i, x_i\}$',
    '分布假设\n$f(y_i \mid x_i, \theta)$',
    '参数化\n$\theta_i = h(x_i, \beta)$',
    '对数似然\n$\ell(\beta)=\sum \ln f$',
    'MLE 估计\n$\hat{\beta}=\arg\max \, \ell$',
    '推断 / 预测 / 比较'
]
xs = np.linspace(0.04, 0.84, len(labels))
width, height = 0.12, 0.28
for i, (x, lab) in enumerate(zip(xs, labels)):
    box = mpatches.FancyBboxPatch((x, 0.36), width, height,
                                  boxstyle='round,pad=0.02,rounding_size=0.02',
                                  facecolor=COLOR_FILL, edgecolor=COLOR_PRIMARY, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + width / 2, 0.50, lab, ha='center', va='center', fontsize=10)
    if i < len(labels) - 1:
        ax.annotate('', xy=(x + width + 0.015, 0.50), xytext=(xs[i + 1] - 0.015, 0.50),
                    arrowprops=dict(arrowstyle='->', color=COLOR_PRIMARY, lw=1.8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
savefig('./figs/method_MLE_fig01_flowchart.png')
print('已生成：./figs/method_MLE_fig01_flowchart.png')


## 2. 图 02：概率 vs. 似然的直觉对比

本图对应讲义第 2 节。左图固定参数，考察样本结果的概率；右图固定样本结果，比较不同参数值的解释能力。二者使用同一个“5 次调查中 3 人步行”的场景，但提问方向完全不同。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
n, p = 5, 0.6
k = np.arange(n + 1)
pmf = stats.binom.pmf(k, n, p)
colors = [COLOR_PRIMARY] * len(k)
colors[3] = COLOR_SECONDARY
axes[0].bar(k, pmf, color=colors, alpha=0.9)
axes[0].set_xlabel('5 次调查中步行人数 k')
axes[0].set_ylabel(r'$P(Y=k\mid \theta=0.6)$')
axes[0].set_title(r'概率视角：$\theta=0.6$ 固定，数据在变')
axes[0].annotate('已观测到', xy=(3, pmf[3]), xytext=(3.7, pmf[3] + 0.08),
                 arrowprops=dict(arrowstyle='->', lw=1.2, color=COLOR_NEUTRAL), fontsize=9)

theta = np.linspace(0.01, 0.99, 500)
L = theta**3 * (1 - theta)**2
axes[1].plot(theta, L, color=COLOR_PRIMARY, lw=2)
axes[1].axvline(0.6, color=COLOR_SECONDARY, lw=1.5, ls='--')
axes[1].set_xlabel(r'$\theta$')
axes[1].set_ylabel(r'$L(\theta)=\theta^3(1-\theta)^2$')
axes[1].set_title(r'似然视角：数据固定，$\theta$ 在变')
axes[1].annotate('最大似然估计值', xy=(0.6, 0.6**3 * 0.4**2), xytext=(0.68, L.max() * 0.9),
                 arrowprops=dict(arrowstyle='->', lw=1.2, color=COLOR_NEUTRAL), fontsize=9)
savefig('./figs/method_MLE_fig02_prob_vs_likelihood.png')
print('已生成：./figs/method_MLE_fig02_prob_vs_likelihood.png')


## 3. 图 03 和图 04：Bernoulli 例子的似然函数与网格搜索

这两张图服务于讲义第 3 节。图 03 展示 Bernoulli 似然函数与对数似然函数的形状；图 04 展示最简单的“网格搜索”思路，帮助学生理解数值优化的直觉。


In [ ]:
theta = np.linspace(0.01, 0.99, 500)
L = theta**3 * (1 - theta)**2
ll = 3 * np.log(theta) + 2 * np.log(1 - theta)

fig, axes = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
axes[0].plot(theta, L, color=COLOR_PRIMARY, lw=2)
axes[0].axvline(0.6, color=COLOR_SECONDARY, lw=1.5, ls='--')
axes[0].scatter([0.6], [0.6**3 * 0.4**2], color=COLOR_SECONDARY, zorder=3)
axes[0].set_ylabel(r'$L(\theta)$')
axes[0].set_title('似然函数')
axes[1].plot(theta, ll, color=COLOR_PRIMARY, lw=2)
axes[1].axvline(0.6, color=COLOR_SECONDARY, lw=1.5, ls='--')
axes[1].scatter([0.6], [3 * np.log(0.6) + 2 * np.log(0.4)], color=COLOR_SECONDARY, zorder=3)
axes[1].set_xlabel(r'$\theta$')
axes[1].set_ylabel(r'$\ell(\theta)$')
axes[1].set_title('对数似然函数')
savefig('./figs/method_MLE_fig03_bernoulli_likelihood.png')
print('已生成：./figs/method_MLE_fig03_bernoulli_likelihood.png')

fig, ax = plt.subplots(figsize=(6, 4))
grid = np.linspace(0.1, 0.9, 17)
grid_ll = 3 * np.log(grid) + 2 * np.log(1 - grid)
ax.plot(theta, ll, color=COLOR_NEUTRAL, lw=1.5, alpha=0.85)
for x, y in zip(grid, grid_ll):
    ax.vlines(x, ymin=grid_ll.min() - 0.15, ymax=y, colors=COLOR_NEUTRAL, lw=0.8, alpha=0.7)
ax.scatter(grid, grid_ll, color=COLOR_PRIMARY, s=35, zorder=3)
max_idx = np.argmax(grid_ll)
ax.scatter([grid[max_idx]], [grid_ll[max_idx]], color=COLOR_SECONDARY, s=70, zorder=4)
ax.set_xlabel(r'$\theta$')
ax.set_ylabel(r'$\ell(\theta)$')
ax.set_title('网格搜索：在参数空间中寻找最优点')
savefig('./figs/method_MLE_fig04_bernoulli_grid.png')
print('已生成：./figs/method_MLE_fig04_bernoulli_grid.png')


## 4. 图 05：参数化变换链

本图对应讲义第 4 节。目的是把“线性预测器 → 连接函数 → 分布参数”这一抽象过程画出来，使学生意识到：Logit、Poisson 和正态线性模型虽然形式不同，但共享同一套参数化骨架。


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axis('off')
row_y = [0.78, 0.50, 0.22]
rows = [
    ('线性预测器：$\eta_i=x_i^\top\beta$\n范围：$(-\infty,+\infty)$', '恒等', '分布参数：$\mu_i\in\mathbb{R}$\n正态均值'),
    ('线性预测器：$\eta_i=x_i^\top\beta$\n范围：$(-\infty,+\infty)$', 'Logistic', '分布参数：$p_i\in(0,1)$\nLogit 概率'),
    ('线性预测器：$\eta_i=x_i^\top\beta$\n范围：$(-\infty,+\infty)$', '指数', '分布参数：$\lambda_i>0$\nPoisson 强度')
]
colors_rows = [COLOR_PRIMARY, COLOR_ORANGE, COLOR_GREEN]
for y, color, row in zip(row_y, colors_rows, rows):
    left, mid, right = row
    b1 = mpatches.FancyBboxPatch((0.05, y - 0.09), 0.28, 0.16, boxstyle='round,pad=0.02', facecolor='white', edgecolor=color, linewidth=1.6)
    b2 = mpatches.FancyBboxPatch((0.67, y - 0.09), 0.26, 0.16, boxstyle='round,pad=0.02', facecolor='white', edgecolor=color, linewidth=1.6)
    ax.add_patch(b1); ax.add_patch(b2)
    ax.text(0.19, y, left, ha='center', va='center', fontsize=9)
    ax.text(0.80, y, right, ha='center', va='center', fontsize=9)
    ax.annotate('', xy=(0.63, y), xytext=(0.35, y), arrowprops=dict(arrowstyle='->', lw=1.8, color=color))
    ax.text(0.49, y + 0.03, mid, ha='center', va='bottom', fontsize=10, color=color)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
savefig('./figs/method_MLE_fig05_parameterization.png')
print('已生成：./figs/method_MLE_fig05_parameterization.png')


## 5. 图 06 和图 07：正态模型中的均值估计

这两张图服务于讲义第 5 节。图 06 展示不同均值参数下，正态密度如何覆盖同一组数据；图 07 则展示对数似然关于均值参数的曲线，说明峰值恰好出现在样本均值处。


In [ ]:
np.random.seed(42)
data = np.random.normal(170, 5, 30)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
xgrid = np.linspace(150, 190, 500)
for mu, col, ls, lw, lab in [
    (160, COLOR_NEUTRAL, '--', 1.5, r'$\mu=160$（偏低）'),
    (170, COLOR_PRIMARY, '-', 2.5, r'$\mu=170$（MLE 估计值）'),
    (180, COLOR_NEUTRAL, '--', 1.5, r'$\mu=180$（偏高）')
]:
    ax.plot(xgrid, stats.norm.pdf(xgrid, mu, 5), color=col, ls=ls, lw=lw, label=lab)
rug_y = np.full_like(data, -0.003)
ax.plot(data, rug_y, '|', color=COLOR_NEUTRAL, markersize=10, markeredgewidth=1.2)
ax.set_ylim(-0.01, stats.norm.pdf(xgrid, 170, 5).max() * 1.15)
ax.set_xlabel('身高')
ax.set_ylabel('密度')
ax.set_title('不同均值设定下的正态密度曲线')
ax.legend(frameon=False, loc='upper left')
savefig('./figs/method_MLE_fig06_normal_fit_mu.png')
print('已生成：./figs/method_MLE_fig06_normal_fit_mu.png')

fig, ax = plt.subplots(figsize=(6, 4))
mu_grid = np.linspace(155, 185, 400)
sigma = 5
loglik_mu = np.array([-(len(data) / 2) * np.log(2 * np.pi * sigma**2) - np.sum((data - mu)**2) / (2 * sigma**2) for mu in mu_grid])
mu_hat = data.mean()
ll_hat = -(len(data) / 2) * np.log(2 * np.pi * sigma**2) - np.sum((data - mu_hat)**2) / (2 * sigma**2)
ax.plot(mu_grid, loglik_mu, color=COLOR_PRIMARY, lw=2)
ax.axvline(mu_hat, color=COLOR_SECONDARY, lw=1.5, ls='--')
ax.scatter([mu_hat], [ll_hat], color=COLOR_SECONDARY, zorder=3)
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\ell(\mu)$')
ax.set_title('对数似然函数关于均值参数的曲线')
savefig('./figs/method_MLE_fig07_loglik_mu_curve.png')
print('已生成：./figs/method_MLE_fig07_loglik_mu_curve.png')


## 6. 图 08：OLS 与 MLE 的等价性

本图继续服务于讲义第 5 节。这里直接画出一维参数下的两条目标函数曲线：一条是 RSS，另一条是负对数似然。两条曲线虽然尺度不同，但极值点位置完全一致。


In [ ]:
np.random.seed(42)
n = 50
x = np.random.normal(0, 1, n)
eps = np.random.normal(0, 1, n)
y = 1 + 2 * x + eps
beta_grid = np.linspace(0, 4, 400)
intercept = 1.0
rss = np.array([np.sum((y - intercept - b * x)**2) for b in beta_grid])
nll = np.array([(n / 2) * np.log(2 * np.pi * 1.0) + np.sum((y - intercept - b * x)**2) / 2 for b in beta_grid])
beta_hat = beta_grid[np.argmin(rss)]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(beta_grid, rss, color=COLOR_PRIMARY, lw=2)
axes[0].axvline(beta_hat, color=COLOR_SECONDARY, lw=1.5, ls='--')
axes[0].set_title('最小化残差平方和（OLS）')
axes[0].set_xlabel(r'$\beta_1$')
axes[0].set_ylabel('RSS')
axes[1].plot(beta_grid, nll, color=COLOR_PRIMARY, lw=2)
axes[1].axvline(beta_hat, color=COLOR_SECONDARY, lw=1.5, ls='--')
axes[1].set_title('最小化负对数似然（MLE）')
axes[1].set_xlabel(r'$\beta_1$')
axes[1].set_ylabel(r'$-\ell(\beta_1)$')
savefig('./figs/method_MLE_fig08_ols_is_mle.png')
print('已生成：./figs/method_MLE_fig08_ols_is_mle.png')


## 7. 生成模拟数据并保存

下面开始生成 3 份可复用的模拟数据，分别对应：正态线性模型、Logit 模型和 Poisson 模型。这些数据既用于 `MLE_case.ipynb`，也可在后续章节中复用。


### 7.1 数据 01：正态线性模型

这份数据用于展示 OLS 与 MLE 的等价性。数据生成过程为

$$
y_i = 1 + 2x_{1i} - 0.5x_{2i} + \varepsilon_i, \qquad \varepsilon_i \sim N(0, 1.5^2)
$$


In [ ]:
np.random.seed(2024)
n = 500
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)
eps = np.random.normal(0, 1.5, n)
y = 1 + 2 * x1 - 0.5 * x2 + eps
df1 = pd.DataFrame({'y': y, 'x1': x1, 'x2': x2})
df1.to_csv('./data/method_MLE_data01_normal_linear.csv', index=False)
print(df1.describe().round(3))


### 7.2 数据 02：Logit 模型

这份数据模拟“信用卡违约”场景。潜变量模型为

$$
y_i^* = -2 + 1.5 \, income_i - 0.8 \, age\_std_i + u_i, \qquad u_i \sim Logistic(0,1)
$$


In [ ]:
np.random.seed(2024)
n = 1000
income = np.random.normal(5, 1, n)
age_std = (np.random.normal(35, 8, n) - 35) / 8
u = np.random.logistic(0, 1, n)
y_star = -2 + 1.5 * income - 0.8 * age_std + u
default = (y_star > 0).astype(int)
df2 = pd.DataFrame({'default': default, 'income': income, 'age_std': age_std})
df2.to_csv('./data/method_MLE_data02_logit.csv', index=False)
print(f'违约率 = {default.mean():.3f}')
print(df2.describe().round(3))


### 7.3 数据 03：Poisson 模型

这份数据模拟“月度交易次数”场景，设

$$
y_i \sim Poisson(\lambda_i), \qquad \ln \lambda_i = 0.5 + 0.3 \, income_i + 0.2 \, experience_i
$$


In [ ]:
np.random.seed(2024)
n = 800
income = np.random.normal(0, 1, n)
experience = np.random.normal(0, 1, n)
lam = np.exp(0.5 + 0.3 * income + 0.2 * experience)
trade_count = np.random.poisson(lam)
df3 = pd.DataFrame({'trade_count': trade_count, 'income': income, 'experience': experience})
df3.to_csv('./data/method_MLE_data03_poisson.csv', index=False)
print(f'交易次数均值 = {trade_count.mean():.3f}，最大值 = {trade_count.max()}')
print(df3.describe().round(3))


## 8. 本笔记小结

到这里，本章主讲义和案例分析所需的核心图形与模拟数据已经全部准备完成。后续若要更新图形风格或调整数据生成过程，只需重新运行本笔记即可。
